In [1]:

import os
import pickle
import warnings
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import multimil as mtm

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    brier_score_loss,
)

warnings.filterwarnings("ignore")
scvi.settings.seed = 0


/home/bin_jip/.local/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Global seed set to 0
Global seed set to 0


In [2]:
@dataclass
class Settings:
    dataset_name: str = "asthma"
    split_number: int = 0

    adata_path: str = "/data2/project/bin_jip/Biomarker/data/asthma/asthma_baseline_data.h5ad"
    splits_directory: str = "/data2/project/bin_jip/Biomarker/data/splits/asthma/baseline/kfold"

    label_column: str = "Response2"
    sample_key: str = "Sample_ID"

    binary_positive_labels: Tuple[str, ...] = ("1",)
    binary_negative_labels: Tuple[str, ...] = ("0",)

    # training
    max_epochs: int = 200
    learning_rate: float = 5e-4
    weight_decay: float = 1e-3
    batch_size: int = 256

    # external early stopping (on query_val)
    early_stop_patience: int = 30
    early_stop_min_delta: float = 0.0
    early_stop_monitor: str = "val_bce"  # "val_bce" or "auprc" or "auroc" or "f1" or "brier"
    early_stop_mode: str = "min"         # "min" for val_bce/brier, "max" for auprc/auroc/f1

    # after choosing best_epoch using val, optionally retrain on TRAIN+VAL to use all samples
    retrain_on_train_val: bool = False

    # model hparams
    dropout: float = 0.2
    scoring: str = "gated_attn"
    attn_dim: int = 16


settings = Settings()

split_path = os.path.join(
    settings.splits_directory,
    f"{settings.dataset_name}_idx_{int(settings.split_number)}.pkl"
)

print("adata_path:", settings.adata_path)
print("split_path:", split_path)


adata_path: /data2/project/bin_jip/Biomarker/data/asthma/asthma_baseline_data.h5ad
split_path: /data2/project/bin_jip/Biomarker/data/splits/asthma/baseline/kfold/asthma_idx_0.pkl


In [3]:
adata = sc.read_h5ad(settings.adata_path)

label_column = str(settings.label_column)
sample_key = str(settings.sample_key)

positive_set = {str(v) for v in settings.binary_positive_labels}
negative_set = {str(v) for v in settings.binary_negative_labels}

def map_binary_labels(values: List[str], positive_labels: set, negative_labels: set) -> List[str]:
    mapped: List[str] = []
    for value in values:
        value_str = str(value)
        if value_str in positive_labels:
            mapped.append("1")
        elif value_str in negative_labels:
            mapped.append("0")
        else:
            raise ValueError(
                f"Label '{value_str}' not in positive={sorted(list(positive_labels))} "
                f"or negative={sorted(list(negative_labels))}"
            )
    return mapped

adata.obs[label_column] = adata.obs[label_column].astype(str)
adata.obs["label_binary"] = pd.Categorical(
    map_binary_labels(adata.obs[label_column].tolist(), positive_set, negative_set),
    categories=["0", "1"],
)

disease_key = "label_binary"

adata.obs[sample_key] = adata.obs[sample_key].astype(str).astype("category")
adata.obs[disease_key] = adata.obs[disease_key].astype("category")

print("n_obs:", adata.n_obs, "n_vars:", adata.n_vars)
print("sample_key:", sample_key, "| disease_key:", disease_key)


n_obs: 259552 n_vars: 14314
sample_key: Sample_ID | disease_key: label_binary


In [ ]:
def load_split_indices(split_file_path: str, adata_obj: sc.AnnData) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if not os.path.exists(split_file_path):
        raise FileNotFoundError(f"Split file not found: {split_file_path}")

    with open(split_file_path, "rb") as handle:
        split_indices = pickle.load(handle)

    if not isinstance(split_indices, list) or len(split_indices) != 3:
        raise ValueError("Split file must be list [train_cells, val_cells, test_cells].")

    split_arrays: List[np.ndarray] = []
    for split_name, raw_values in zip(["train", "val", "test"], split_indices):
        array = np.asarray(raw_values, dtype=np.int64).reshape(-1)
        if array.size > 0 and int(array.min()) < 0:
            raise ValueError(f"Split '{split_name}' includes negative indices.")
        split_arrays.append(array)

    # bounds
    maximum_index = int(max((int(a.max()) for a in split_arrays if a.size > 0), default=-1))
    if maximum_index >= int(adata_obj.n_obs):
        raise IndexError(
            f"Split indices out of bounds (max={maximum_index}, n_obs={adata_obj.n_obs}). "
            "Make sure adata matches the one used to create split."
        )

    # duplicates across splits
    merged = np.concatenate(split_arrays, axis=0) if any(a.size > 0 for a in split_arrays) else np.zeros((0,), dtype=np.int64)
    if int(np.unique(merged).size) != int(merged.size):
        raise ValueError("Split indices contain duplicates across train/val/test.")

    return split_arrays[0], split_arrays[1], split_arrays[2]

train_index, val_index, test_index = load_split_indices(split_path, adata)

adata_train = adata[train_index]
adata_val = adata[val_index]
adata_test = adata[test_index]

print("train cells:", adata_train.n_obs, "| val cells:", adata_val.n_obs, "| test cells:", adata_test.n_obs)
print("train samples:", adata_train.obs[sample_key].nunique(),
      "| val samples:", adata_val.obs[sample_key].nunique(),
      "| test samples:", adata_test.obs[sample_key].nunique())


train cells: 172134 | val cells: 58129 | test cells: 29289
train samples: 17 | val samples: 6 | test samples: 5


In [5]:
def extract_bag_predictions(
    query_adata: sc.AnnData,
    disease_key_name: str,
    sample_key_name: str
) -> Tuple[pd.DataFrame, List[str]]:
    bag_key = f"bag_full_predictions_{disease_key_name}"
    true_key = f"bag_true_{disease_key_name}"

    if bag_key not in query_adata.uns:
        raise KeyError(f"Missing '{bag_key}' in query_adata.uns. Did you call get_model_output(adata=...)?")
    if true_key not in query_adata.uns:
        raise KeyError(f"Missing '{true_key}' in query_adata.uns. Did you call get_model_output(adata=...)?")
    if "bags" not in query_adata.obs:
        raise KeyError("Missing 'bags' in query_adata.obs. get_model_output() should populate it.")

    bag_full = query_adata.uns[bag_key]
    if isinstance(bag_full, pd.DataFrame):
        class_names = [str(c) for c in bag_full.columns]
        prob_matrix = bag_full.values
    else:
        prob_matrix = np.asarray(bag_full)
        if pd.api.types.is_categorical_dtype(query_adata.obs[disease_key_name]):
            class_names = [str(v) for v in query_adata.obs[disease_key_name].cat.categories]
        else:
            class_names = [str(v) for v in sorted(pd.unique(query_adata.obs[disease_key_name].astype(str)))]
        if len(class_names) != int(prob_matrix.shape[1]):
            class_names = [str(i) for i in range(int(prob_matrix.shape[1]))]

    bag_true = [str(v) for v in query_adata.uns[true_key]]

    bag_map = (
        query_adata.obs[[sample_key_name, "bags"]]
        .drop_duplicates()
        .sort_values("bags")
        .reset_index(drop=True)
    )

    prob_df = pd.DataFrame(prob_matrix, columns=[f"prob_{c}" for c in class_names])
    out = pd.concat([bag_map, prob_df], axis=1)

    out["true_label"] = bag_true[: len(out)]
    pred_index = prob_matrix.argmax(axis=1)
    out["pred_label"] = [class_names[int(i)] for i in pred_index]

    return out, class_names

def evaluate_binary_from_bags(
    bag_df: pd.DataFrame,
    class_names: List[str],
    positive_label: str = "1"
) -> Dict[str, float]:
    positive_label = str(positive_label)
    if positive_label not in class_names:
        raise ValueError(f"positive_label='{positive_label}' not in class_names={class_names}")

    prob_positive = bag_df[f"prob_{positive_label}"].astype(float).to_numpy()
    true_binary = (bag_df["true_label"].astype(str).to_numpy() == positive_label).astype(int)
    pred_binary = (prob_positive >= 0.5).astype(int)

    metrics: Dict[str, float] = {}
    try:
        metrics["auroc"] = float(roc_auc_score(true_binary, prob_positive))
    except Exception:
        metrics["auroc"] = float("nan")
    try:
        metrics["auprc"] = float(average_precision_score(true_binary, prob_positive))
    except Exception:
        metrics["auprc"] = float("nan")

    metrics["f1"] = float(f1_score(true_binary, pred_binary))
    metrics["brier"] = float(brier_score_loss(true_binary, prob_positive))
    return metrics


def binary_cross_entropy_from_probs(true_binary: np.ndarray, prob_positive: np.ndarray, epsilon: float = 1e-8) -> float:
    prob_positive = np.clip(prob_positive.astype(float), epsilon, 1.0 - epsilon)
    true_binary = true_binary.astype(float)
    loss = -(true_binary * np.log(prob_positive) + (1.0 - true_binary) * np.log(1.0 - prob_positive))
    return float(np.mean(loss))


In [ ]:
# Reference (train) and queries (val/test)
print("* Initializing MIL model and preparing data...")
ref_train = adata_train
print("train")
query_val = adata_val
print("val")
query_test = adata_test
print("test")
# sort for stable bag ordering
ref_train = ref_train[ref_train.obs[sample_key].sort_values().index]
query_val = query_val[query_val.obs[sample_key].sort_values().index]
query_test = query_test[query_test.obs[sample_key].sort_values().index]

classification_keys = [disease_key]
categorical_covariate_keys = [disease_key, sample_key]  # MultiMIL requires sample_key registered as categorical
print("**")
mtm.model.MILClassifier.setup_anndata(
    ref_train,
    categorical_covariate_keys=categorical_covariate_keys,
)
print("**")
mil = mtm.model.MILClassifier(
    ref_train,
    classification=classification_keys,
    sample_key=sample_key,
    class_loss_coef=1.0,
    dropout=float(settings.dropout),
    scoring=str(settings.scoring),
    attn_dim=int(settings.attn_dim),
)

print("MIL model initialized.")

* Initializing MIL model and preparing data...
train
val
test
**


In [ ]:
def setup_query_like_reference(query_adata: sc.AnnData, reference_model: mtm.model.MILClassifier) -> sc.AnnData:
    registry = reference_model.adata_manager.registry

    setup_args = {}
    if isinstance(registry, dict):
        if "setup_args" in registry and isinstance(registry["setup_args"], dict):
            setup_args = registry["setup_args"]
        elif "setup_method_args" in registry and isinstance(registry.get("setup_method_args", None), dict):
            setup_args = registry["setup_method_args"]

    mtm.model.MILClassifier.setup_anndata(
        query_adata,
        source_registry=registry,
        extend_categories=True,
        allow_missing_labels=True,
        **setup_args,
    )
    return query_adata

query_val = setup_query_like_reference(query_val, mil)
query_test = setup_query_like_reference(query_test, mil)

print("query_val/query_test set up with reference registry.")


query_val/query_test set up with reference registry.


In [ ]:
import numpy as np
import torch
from pytorch_lightning.callbacks import Callback
from multimil.dataloaders import GroupAnnDataLoader


def _extract_loss_tensor(validation_step_output):
    if torch.is_tensor(validation_step_output):
        return validation_step_output
    if isinstance(validation_step_output, dict):
        for key in ["loss", "validation_loss", "val_loss", "loss_validation"]:
            if key in validation_step_output and torch.is_tensor(validation_step_output[key]):
                return validation_step_output[key]
        for value in validation_step_output.values():
            if torch.is_tensor(value):
                return value
    raise TypeError(f"Cannot extract loss from: {type(validation_step_output)}")

class ExternalValOriginalLossEarlyStop(Callback):
    """
    - query_val에서 '원래 MultiMIL loss'를 계산(AdversarialTrainingPlan.validation_step)
    - 그 loss로 best/patience early stopping
    - best model state_dict 저장/복원
    """
    def __init__(
        self,
        mil_model,
        query_validation_adata,
        batch_size: int = 256,
        patience: int = 30,
        min_delta: float = 0.0,
        verbose: bool = True,
    ):
        super().__init__()
        self.mil_model = mil_model
        self.query_validation_adata = query_validation_adata
        self.batch_size = int(batch_size)
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.verbose = bool(verbose)

        self.best_val_loss = None
        self.best_epoch = None
        self.wait_count = 0
        self.best_state_dict_cpu = None

    @torch.no_grad()
    def _compute_val_loss(self, training_plan_module) -> float:
        # query_val을 bag grouping 그대로 loader로 만들고, validation_step을 직접 호출
        validation_loader = self.mil_model._make_data_loader(
            adata=self.query_validation_adata,
            batch_size=self.batch_size,
            min_size_per_class=self.batch_size,
            data_loader_class=GroupAnnDataLoader,
            shuffle=False,
            shuffle_classes=False,
            group_column=self.mil_model.sample_key,
            drop_last=False,
        )

        device = training_plan_module.device
        loss_values = []

        training_plan_module.eval()
        for batch_index, tensor_dictionary in enumerate(validation_loader):
            tensor_dictionary_on_device = {}
            for key, value in tensor_dictionary.items():
                tensor_dictionary_on_device[key] = value.to(device) if torch.is_tensor(value) else value

            output = training_plan_module.validation_step(tensor_dictionary_on_device, batch_index)
            loss_tensor = _extract_loss_tensor(output)
            loss_values.append(float(loss_tensor.detach().cpu().item()))

        return float(np.mean(loss_values)) if len(loss_values) > 0 else float("nan")

    def on_train_epoch_end(self, trainer, training_plan_module):
        epoch_index = int(trainer.current_epoch)
        val_loss = self._compute_val_loss(training_plan_module)

        if self.verbose:
            print(f"[epoch {epoch_index:03d}] external_val_loss(original)={val_loss:.6f}")

        improved = False
        if self.best_val_loss is None or np.isnan(self.best_val_loss):
            improved = True
        else:
            improved = val_loss < (self.best_val_loss - self.min_delta)

        if improved:
            self.best_val_loss = float(val_loss)
            self.best_epoch = epoch_index
            self.wait_count = 0

            current_state = self.mil_model.module.state_dict()
            self.best_state_dict_cpu = {k: v.detach().cpu().clone() for k, v in current_state.items()}

            if self.verbose:
                print(f"  -> best updated: {self.best_val_loss:.6f} (epoch {self.best_epoch})")
        else:
            self.wait_count += 1
            if self.verbose:
                print(f"  -> no improvement (wait {self.wait_count}/{self.patience})")

            if self.wait_count >= self.patience:
                if self.verbose:
                    print(f"  -> early stopping triggered at epoch {epoch_index}")
                trainer.should_stop = True

    def restore_best_weights(self):
        if self.best_state_dict_cpu is None:
            raise RuntimeError("No best weights stored.")
        device = next(self.mil_model.module.parameters()).device
        state_dict_on_device = {k: v.to(device) for k, v in self.best_state_dict_cpu.items()}
        self.mil_model.module.load_state_dict(state_dict_on_device)


In [ ]:
external_val_callback = ExternalValOriginalLossEarlyStop(
    mil_model=mil,
    query_validation_adata=query_val,
    batch_size=settings.batch_size,
    patience=settings.early_stop_patience,
    min_delta=settings.early_stop_min_delta,
    verbose=True,
)

mil.train(
    max_epochs=int(settings.max_epochs),
    lr=float(settings.learning_rate),
    train_size=1.0,
    validation_size=0.0,     # 내부 random val OFF
    early_stopping=False,    # 내부 early stop OFF
    save_best=False,         # 내부 SaveBestState OFF
    batch_size=int(settings.batch_size),
    weight_decay=float(settings.weight_decay),
    use_gpu=None,
    callbacks=[external_val_callback],
    logger=False,
)

# 학습 종료 후 best weight 복원
external_val_callback.restore_best_weights()
print("Best epoch:", external_val_callback.best_epoch, "Best val loss:", external_val_callback.best_val_loss)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Epoch 1/200:   0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:
mil.get_model_output(adata=query_test, batch_size=int(settings.batch_size))

test_bag_df, test_class_names = extract_bag_predictions(
    query_adata=query_test,
    disease_key_name=disease_key,
    sample_key_name=sample_key,
)

test_metrics = evaluate_binary_from_bags(test_bag_df, test_class_names, positive_label="1")
print("\n===== TEST METRICS =====")
print("AUROC:", test_metrics["auroc"])
print("AUPRC:", test_metrics["auprc"])
print("F1   :", test_metrics["f1"])
print("Brier:", test_metrics["brier"])

# sample-wise table (query/sample id, prob_1, true, pred)
display_columns = [sample_key, "true_label", "pred_label"] + [c for c in test_bag_df.columns if c.startswith("prob_")]
test_display_df = test_bag_df[display_columns].sort_values(sample_key).reset_index(drop=True)

print("\n===== PER-SAMPLE PREDICTIONS (HEAD) =====")
print(test_display_df.head(30))

output_csv_path = f"multimil_test_predictions_{settings.dataset_name}_split{int(settings.split_number)}.csv"
test_display_df.to_csv(output_csv_path, index=False)
print("\nSaved:", output_csv_path)